# Лабораторная 1: SimpleRNN для задачи many-to-one (решение)

## Цель
Реализовать рекуррентную модель, которая по всей последовательности выдает один бинарный ответ.

Формат `many-to-one` означает: много временных шагов на входе и один итоговый выход на последовательность.

## Контракт данных
Используется учебная синтетическая постановка:
- входная последовательность содержит значения `-1` и `+1`;
- целевая метка равна `1`, если сумма элементов последовательности положительна, иначе `0`.

Задача требует учета контекста по всей длине последовательности, поэтому используется рекуррентная сеть.


## Таблица форм тензоров

| Тензор | Форма | Смысл | Где используется |
|---|---|---|---|
| `X` | `(N, T, 1)` | Полный набор входных последовательностей | Генерация данных |
| `y` | `(N,)` | Бинарные целевые метки | Генерация данных |
| `X_train`, `y_train` | `(N_train, T, 1)`, `(N_train,)` | Обучающая часть | `model.fit` |
| `X_test`, `y_test` | `(N_test, T, 1)`, `(N_test,)` | Контрольная часть | `model.evaluate` |
| `probs` | `(N_test,)` | Вероятности класса `1` | `model.predict` |
| `preds` | `(N_test,)` | Бинарные предсказания после порога | Оценка качества |


## Контракт модели
- На вход `model.fit` подается `X_train` формы `(batch, time, features)` и `y_train`.
- Модель возвращает вероятность положительного класса для каждой последовательности.
- На выходе `model.predict(X_test)` получается вектор вероятностей, который преобразуется в классы порогом `0.5`.
- Функция потерь: бинарная кросс-энтропия (`binary_crossentropy`).


## Мини-теория
Прямой проход (forward pass) для `SimpleRNN`:

$$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h), \quad
s = W_{hy}h_T + b_y, \quad
\hat{y}=\sigma(s)
$$

Итоговый логит `s` вычисляется только из последнего скрытого состояния `h_T`, что соответствует формату `many-to-one`.


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(42)
print('Версия TensorFlow:', tf.__version__)

## Генерация данных
**Что сделать:** сформировать входные последовательности и бинарные метки.

**Почему:** модель должна учиться извлекать итоговый признак из всей последовательности.

**Ожидаемые формы:** `X -> (N,T,1)`, `y -> (N,)`.

**Частая ошибка:** суммирование не по временной оси или некорректная форма `y`.


In [ ]:
def make_dataset(n_samples: int = 6000, seq_len: int = 12):
    x = np.random.choice([-1.0, 1.0], size=(n_samples, seq_len, 1)).astype(np.float32)
    sums = x.sum(axis=1).reshape(-1)
    y = (sums > 0).astype(np.float32)
    return x, y


X, y = make_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Форма X_train:', X_train.shape)
print('Форма y_train:', y_train.shape)
print('Форма X_test :', X_test.shape)
print('Форма y_test :', y_test.shape)

In [ ]:
# Мини-проверка данных
assert X.ndim == 3 and X.shape[-1] == 1, 'Ожидается X формы (N,T,1)'
assert y.ndim == 1, 'Ожидается y формы (N,)'
assert set(np.unique(y)).issubset({0.0, 1.0}), 'Метки должны быть бинарными'
print('Мини-проверка данных: OK')

## Модель
**Что сделать:** собрать `Sequential`-модель через `add(...)`.

**Почему:** для `many-to-one` нужен один итоговый выход.

**Ожидаемая форма выхода:** `(batch, 1)`.

**Частая ошибка:** отсутствие компиляции модели перед обучением.


In [ ]:
def build_model(input_shape):
    model = tf.keras.Sequential(name='simple_rnn_many_to_one')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.SimpleRNN(24))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


model = build_model(X_train.shape[1:])
model.summary()

In [ ]:
# Мини-проверка модели
assert model.output_shape[-1] == 1, 'Выходной слой должен иметь 1 нейрон'
print('Мини-проверка модели: OK')

## Трассировка одного примера через модель
Проверка показывает формы тензоров в трех точках:
1. входной пример;
2. выход рекуррентного слоя;
3. выходного слоя (вероятность класса).


In [ ]:
sample_x = X_train[:1]
_ = model(sample_x)

# Выход после рекуррентного слоя
rnn_features_model = tf.keras.Model(inputs=model.inputs[0], outputs=model.layers[-2].output)
rnn_features = rnn_features_model.predict(sample_x, verbose=0)

# Выход модели
sample_prob = model.predict(sample_x, verbose=0)

print('Вход sample_x           :', sample_x.shape)
print('После рекуррентного слоя:', rnn_features.shape)
print('После выходного слоя    :', sample_prob.shape)
print('Вероятность класса 1    :', float(sample_prob[0, 0]))

## Как идет обучение внутри эпохи
Обучение выполняется по мини-батчам (mini-batch) и включает цикл:
1. прямой проход (`forward pass`) через все шаги времени;
2. вычисление функции потерь;
3. обратное распространение через время (`BPTT`);
4. обновление параметров оптимизатором.

Для `SimpleRNN` одна память `h_t` переносится по времени без дополнительных ворот, поэтому на длинных зависимостях возможны затухающие или взрывающиеся градиенты.


## Обучение
**Что сделать:** запустить `fit` на обучающей части с `validation_split=0.2`.

**Почему:** валидационная часть нужна для контроля обобщающей способности во время обучения, а `test` сохраняется для финальной проверки.


In [ ]:
def train_model(model, X_train, y_train, epochs: int = 10):
    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=64,
        verbose=0,
    )
    return history


history = train_model(model, X_train, y_train)

In [ ]:
# Мини-проверка обучения
assert 'loss' in history.history and 'val_loss' in history.history
assert 'accuracy' in history.history and 'val_accuracy' in history.history
print('Последняя val_accuracy:', float(history.history['val_accuracy'][-1]))
print('Мини-проверка обучения: OK')

## Оценка и диагностика
На этом этапе вычисляются:
- значение функции потерь на `test`;
- точность (`accuracy`) на `test`;
- ручная проверка точности после порогового преобразования вероятностей.


In [ ]:
def evaluate_model(model, X_test, y_test):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    probs = model.predict(X_test, verbose=0).reshape(-1)
    preds = (probs >= 0.5).astype(np.float32)
    acc_manual = (preds == y_test).mean()
    return {
        'loss': float(loss),
        'accuracy': float(acc),
        'accuracy_manual': float(acc_manual),
        'probs': probs,
        'preds': preds,
    }


metrics = evaluate_model(model, X_test, y_test)
print({k: v for k, v in metrics.items() if k in ('loss', 'accuracy', 'accuracy_manual')})

In [ ]:
# Мини-проверка метрик
assert 0.0 <= metrics['accuracy'] <= 1.0
assert abs(metrics['accuracy'] - metrics['accuracy_manual']) < 0.02
if metrics['accuracy'] >= 0.90:
    print('Целевой порог достигнут: accuracy >= 0.90')
else:
    print('Целевой порог не достигнут: проверьте данные, модель и параметры обучения')

## Что ожидать на практике
- В начале обучения `loss` быстро снижается, затем стабилизируется.
- Если `train_accuracy` растет, а `val_accuracy` не растет, вероятно переобучение.
- Если обе точности низкие, вероятно недообучение или ошибка в разметке/формах данных.
- Для этой постановки при корректной реализации обычно достигается `accuracy >= 0.90`.


In [ ]:
# Визуализация динамики обучения
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Динамика функции потерь')
plt.xlabel('Эпоха')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.title('Динамика точности')
plt.xlabel('Эпоха')
plt.legend()

plt.tight_layout()
plt.show()

## Вопросы для самопроверки
1. Почему для `many-to-one` используется только финальное состояние `h_T`?
2. Что означает форма входа `(batch, time, features)` в терминах задачи?
3. Как интерпретировать разрыв между `train_accuracy` и `val_accuracy`?
4. Зачем нужна отдельная контрольная выборка `test`?


## Типичные ошибки (симптом -> причина -> исправление)
- `accuracy` около `0.5` -> неверная разметка классов -> проверить правило `sum > 0`.
- Ошибка формы тензоров -> перепутан порядок осей -> привести вход к `(N,T,1)`.
- Обучение нестабильно -> слишком большой шаг обучения (`learning rate`) -> уменьшить шаг или проверить оптимизатор.
- Слабая валидация при высоком train -> переобучение -> уменьшить модель или число эпох.
